# Lab 5: Image Classification in Pytorch - Convnets [CIFAR-10]
 **See handout-06:** https://frcs.github.io/4C16-LectureNotes/

Similarly to lab 4, you will train a Neural Network to do classification on CIFAR-10.

This time you will have to use convolutional layers.

the aim of this lab is for you to:
1. become familiar with PyTorch
2. experiment with as many concepts as possible.




# 📘 Lab 5 study guide: convolutional neural networks (handouts 06 & 07)

**Big picture.** In Lab 4, a dense network flattened each image into 3,072 numbers and plateaued around 55% on CIFAR-10. A **convolutional neural network (CNN)** keeps the image as a 2-D grid and applies small learned filters that slide across it. The same edge or texture detector is reused at every position. This is far more parameter-efficient and builds in the right prior for images. **Goal: ≥ 80% accuracy, with ≤ 5M parameters.**

**Concept map**

| Concept | Handout | Where in this notebook |
|---|---|---|
| Why dense layers explode on images | 06 slides 4–6 | compare with Lab 4 |
| Local connectivity and weight sharing | 06 slides 7–8 | `nn.Conv2d` |
| Convolution formula, padding, stride | 06 slides 9–22 | `Conv2d(..., 3, padding=1)` |
| Conv2d parameter count | 06 slide 15 | Section 6 summary |
| Max pooling | 06 slides 23–25 | `nn.MaxPool2d(2)` |
| Architecture design (VGG-style blocks) | 06 slides 30–35 | three conv blocks |
| What filters learn (visualisation) | 06 slides 37–40 | edges → textures → parts |
| Batch normalisation | 07 slides 13–14 | `nn.BatchNorm2d` |
| SGD + momentum, weight decay | 05 slides 51, 54; 07 slide 31 | optimiser cell |
| Dropout | 05 slides 56–57 | `nn.Dropout(0.3)` |
| Improvements: augmentation, ResNet, LR schedules | 07 slides 16–33 | "How to improve" section |

**What you actually have to do.**
1. Write `ConvNN` in `models.py`. It is now done: a VGG-style network with 2,198,602 parameters.
2. Choose the optimiser and hyperparameters (Section 8).
3. Train, check that validation accuracy is above 80%, and save `model.pth`.

The two existing "Explanation" cells (after Sections 5 and 8) describe the architecture and the optimiser. The notes added below supply the theory, the shape and parameter arithmetic, and practical tips.

## Note: Enabling GPU  

By default, Colab runs CPU only. To change to GPU mode, go to Runtime>Change runtime type, select "hardware accelerator" to GPU.

This will restart your instance (obviously), so everytime you do this you'll need to start your again from the start of the notebook.

Beware that GPU instances are not always available. Also, Google might restrict your usage. Google doesn't have clear rules on this.

So be mindful and **only use GPU instances for training. Use CPU only for development and tinkering.**





In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/MyDrive/4c16-labs/code/dist/lab-05/

### 💡 Runtime and Drive set-up

- Use a **GPU runtime** for training. A CNN performs many convolutions per image: CPU training for 30 epochs takes hours, while a GPU takes minutes.
- Unlike Lab 4, this mount cell has no `force_remount` or `try/except`, so re-running it after Drive is already mounted only prints a message. That is harmless.
- `%cd` into `lab-05` makes `import models` load **your** `ConvNN`. Both `torch.save` (Section 10) and `torch.load` (Section 11) then refer to `model.pth` in this folder.
- **Running locally?** Skip this cell, `cd` into `lab-05` yourself, and change the hard-coded `/content/gdrive/...` path in Section 10.

# 1. Import Required Libraries

In [ ]:
import os
import pickle                                     # Used for unpacking our imported python dataset
import importlib
import urllib.request                             # Used for fetching data from URLs
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.ticker import MaxNLocator
import numpy as np
from tqdm.auto import tqdm                        # Used for progress bars
import sklearn.model_selection                    # Helper function for splitting datasets
from IPython.display import clear_output

import torch                                      # The main PyTorch library
import torch.nn as nn                             # Contains pytorch network building blocks (e.g., layers)
import torch.nn.functional as F                   # Contains functions for neural network operations (e.g., activation functions)
from torch.utils.data import Dataset, DataLoader  # Utilities for datasets and data loaders
!pip install torchinfo                            # install missing torchinfo
from torchinfo import summary                     # Used for printing PyTorch model summary

# 2. Loading the Dataset

To load the dataset, the standard practice is to create a `Dataset` class.

In [ ]:
class CIFAR10Dataset(Dataset):
    """Custom CIFAR-10 dataset from provided pickle file.
    Handles download (if missing), train/val split, normalization, tensor conversion.
    """
    def __init__(self, url, pkl_path, train=True, test_size=0.1, random_state=42):
        super().__init__()
        data_dir = os.path.dirname(pkl_path)
        os.makedirs(data_dir, exist_ok=True)
        if not os.path.exists(pkl_path):
            print(f'Downloading dataset to {pkl_path} ...')

            urllib.request.urlretrieve(url, pkl_path)
            print('Download complete.')
        print('Loading dataset...')
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)
        print('Loaded.')
        # loading the input images
        # images are rescaled from 0..255 to 0..1
        X = data['X'].astype('float32') / 255.0
        # loading the Ground Truth classes
        Y = data['Y'].astype('int64')
        if Y.ndim > 1:  # ensure labels are 1D
            Y = np.squeeze(Y)
        self.labels = data['labels']
        self.num_classes = len(self.labels)
        # random split of the data into training set and validation set
        X_train, X_val, y_train, y_val = sklearn.model_selection.train_test_split(
            X, Y, test_size=test_size, random_state=random_state
        )
        # casting numpy data to torch data
        if train:
            self.X = torch.from_numpy(X_train)
            self.y = torch.from_numpy(y_train)
        else:
            self.X = torch.from_numpy(X_val)
            self.y = torch.from_numpy(y_val)
        self.y = self.y.long().view(-1)
        # permuting dimensions to PyTorch conventions (N,C,H,W)
        self.X = self.X.permute(0, 3, 1, 2)
        self.len = self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    def __len__(self):
        return self.len

# 3. Instantiate the training and validation dataset classes

In [ ]:
# Path to the classification dataset
datset_url = 'https://tcddeeplearning.blob.core.windows.net/deeplearning202324/cifar10-dataset.pkl'
# Path to your lab 4 Drive directory on the Colab VM
lab_5_dir = '/home/tcd/Downloads/cifar10-dataset.pkl'

trainset = CIFAR10Dataset(datset_url, lab_5_dir, train=True)
valset = CIFAR10Dataset(datset_url, lab_5_dir, train=False)
print('Train size:', len(trainset), ' Val size:', len(valset))


💡 **Data pipeline: identical to Lab 4.** Pixels are rescaled to [0, 1], the data is split 45,000 train / 5,000 validation with seed 42, and tensors use the `(N, C, H, W)` layout. See Lab 4 for the line-by-line explanation.

**One difference:** this notebook's training loop has **no horizontal-flip augmentation**, so regularisation relies entirely on BatchNorm, dropout and weight decay.

**Why CNNs need the `(C, H, W)` layout.** `nn.Conv2d` slides its filters over the last two dimensions (height and width), and each filter combines **all** input channels. Unlike Lab 4, there is no `Flatten` until the very end of the network.

# 4. Visualize Sample Images

In [ ]:
plt.figure(figsize=(6,4))
for i in range(24):
    img, lab = trainset[i]
    plt.subplot(4,6,i+1)
    plt.imshow(img.permute(1,2,0))
    plt.axis('off')
    plt.title(trainset.labels[lab])
plt.tight_layout()
plt.show()


# same as in lab 4

# 5. Now, go to `models.py` and define your own CNN model here

You will define your own Convolutional Neural Network model using PyTorch's `nn.Module`.

For a comprehensive list of PyTorch building blocks, see: https://docs.pytorch.org/docs/stable/nn.html



In [ ]:
# Note that the new model is called ConvNN and not SimpleNN as in lab 4

import models
from models import ConvNN

### 💡 Why convolutions? Local connectivity and weight sharing (handout-06 slides 4–9)

**The dense-layer problem.** Suppose you wanted a dense layer that maps a 3×32×32 image to 64 feature maps of size 32×32 (which is what `conv1` produces). It would need
$$3072 \times (64\cdot 32\cdot 32) + 65{,}536 \approx \mathbf{201\ million}\ \text{parameters}.$$
`conv1` does it with **1,792**.

**The two ideas that make this possible**
1. **Local connectivity.** Each output pixel looks only at a small 3×3 neighbourhood. Nearby pixels are strongly related; distant ones much less so.
2. **Weight sharing.** The *same* 3×3×C filter is applied at every position. A feature detector, such as a vertical edge detector, is useful everywhere in the image.

Together these make convolution **translation-equivariant**: if the object shifts, its feature map shifts with it, so the network does not have to re-learn each object at every position (as the Lab 4 MLP had to).

**The convolution formula** (handout-06 slide 9), for output channel $k$:
$$y_k(i,j) = b_k + \sum_{c=1}^{C_\text{in}}\sum_{u=-1}^{1}\sum_{v=-1}^{1} w_{k,c}(u,v)\;x_c(i+u,\ j+v)$$

**Output size** for input size $n$, kernel $k$, padding $p$ and stride $s$ (handout-06 slides 12–22):
$$n_\text{out} = \left\lfloor\frac{n + 2p - k}{s}\right\rfloor + 1$$
With $k = 3$, $p = 1$, $s = 1$ this gives $n_\text{out} = n$ ("same" padding). Only `MaxPool2d(2)` halves the size: it keeps the maximum of each 2×2 window and has no parameters (handout-06 slides 23–24).

### Explanation: CNN architecture (`models.py`)

**Objective.** Define a convolutional neural network, `ConvNN`, in `models.py` that classifies 32×32 RGB CIFAR-10 images into 10 classes, using no more than 5M trainable parameters.

**What was implemented.** `ConvNN.__init__` (the layers) and `ConvNN.forward` (the forward pass). The network has three convolutional blocks, each of the form

`Conv3×3 → BatchNorm → ReLU → Conv3×3 → BatchNorm → ReLU → MaxPool(2) → Dropout(0.3)`

with 64, 128 and 256 channels. The spatial size shrinks 32 → 16 → 8 → 4, so the final feature map (256×4×4 = 4096 values) is flattened and passed to `fc1` (4096 → 256, ReLU, dropout) and then `fc2` (256 → 10). `fc2` outputs raw logits, because the softmax is applied inside `nn.CrossEntropyLoss`.

**Method.** A convolutional layer slides a small learned filter over the image and computes, for output channel $k$:
$$y_k(i,j) = b_k + \sum_{c}\sum_{u,v} w_{k,c}(u,v)\,x_c(i+u,\,j+v)$$
`padding=1` keeps the height and width unchanged, so only max-pooling reduces the resolution. Two stacked 3×3 convolutions give a 5×5 receptive field with fewer parameters than one 5×5 filter. Each pooling stage lets deeper layers see larger regions of the image, so features progress from edges, to textures, to object parts. Batch normalisation standardises each channel over the mini-batch and then applies a learned scale and shift:
$$\hat{x} = \frac{x-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}, \qquad y = \gamma\hat{x} + \beta$$
ReLU, $\max(0, x)$, adds the non-linearity.

**Why it matters.** Convolutions share weights across positions and only connect to local neighbourhoods. This makes the model translation-equivariant and needs far fewer parameters than a fully connected network on raw pixels, which is why it generalises better than the lab 4 model. Batch normalisation stabilises training and allows a relatively high learning rate. Dropout randomly zeroes activations during training (with $p = 0.3$, scaling the remaining activations by $1/(1-p)$), which reduces co-adaptation and overfitting. This is important here because no data augmentation is used.

**How the notebook uses it.** This cell imports `ConvNN`. Section 6 reloads it, prints a `torchinfo` summary for an input of shape `(64, 3, 32, 32)`, and checks the trainable parameter count: 2,198,602, below the 5M limit. The largest layer is `fc1`, with about 1.05M weights. The training loop calls `model.train()` and `model.eval()`, so dropout is only active during training and batch normalisation uses its running statistics at evaluation time.

### 🛠️ Shape, parameter and receptive-field walk-through for `ConvNN`

**Parameters.** For `Conv2d(C_in, C_out, k)` the count is $C_\text{in}\cdot C_\text{out}\cdot k^2 + C_\text{out}$ (handout-06 slide 15). For `BatchNorm2d(C)` it is $2C$, one $\gamma$ and one $\beta$ **per channel**.

**Receptive field (RF).** This is the size of the input patch that one output unit "sees". A 3×3 convolution adds $2\times$(current jump) to the RF, and a 2×2 pool adds $1\times$jump and doubles the jump.

| Layer | Output shape `(C, H, W)` | Parameters | RF (pixels) |
|---|---|---|---|
| input | (3, 32, 32) | – | 1 |
| conv1 3→64 + bn1 + ReLU | (64, 32, 32) | 1,792 + 128 | 3 |
| conv2 64→64 + bn2 + ReLU | (64, 32, 32) | 36,928 + 128 | 5 |
| MaxPool + Dropout | (64, 16, 16) | 0 | 6 |
| conv3 64→128 + bn3 + ReLU | (128, 16, 16) | 73,856 + 256 | 10 |
| conv4 128→128 + bn4 + ReLU | (128, 16, 16) | 147,584 + 256 | 14 |
| MaxPool + Dropout | (128, 8, 8) | 0 | 16 |
| conv5 128→256 + bn5 + ReLU | (256, 8, 8) | 295,168 + 512 | 24 |
| conv6 256→256 + bn6 + ReLU | (256, 8, 8) | 590,080 + 512 | 32 |
| MaxPool + Dropout | (256, 4, 4) | 0 | 36 |
| flatten | 4096 | 0 | |
| fc1 4096→256 + ReLU + Dropout | 256 | 1,048,832 | whole image |
| fc2 256→10 (logits) | 10 | 2,570 | |
| **Total** | | **2,198,602** | |

**What the table teaches**
- **Channels go up while the resolution goes down** (64→128→256 channels, 32→16→8→4 pixels). Each block trades spatial detail for richer features: edges, then textures, then object parts (handout-06 slides 37–40). This is the VGG design pattern (handout-06 slides 32–35).
- After conv6, each unit sees about **32×32 pixels, i.e. the whole image**. Stacking small 3×3 filters with pooling builds a large receptive field cheaply.
- **The CNN has fewer parameters than the Lab 4 MLP** (2.2M vs 3.8M) yet should reach much higher accuracy, because the parameters are used efficiently.
- Almost half of the parameters (1.05M) are in `fc1`. Replacing flatten + `fc1` with **global average pooling** (`nn.AdaptiveAvgPool2d(1)` → `Linear(256, 10)`) would remove most of them, which is what modern networks do.
- `torch.flatten(x, 1)` keeps the batch dimension and flattens `(256, 4, 4)` into 4096. This is why `fc1` must be `nn.Linear(256*4*4, ...)`. If you change the number of pools or channels, recompute this number (or let the shape-tracing helper below tell you).

**How to write your own `ConvNN`.** Repeat the pattern `[Conv3×3 → BN → ReLU] ×2 → MaxPool → Dropout` for 2–4 blocks, doubling the channels each time. Then flatten and add one or two `Linear` layers, ending with `Linear(..., num_classes)` with **no softmax**. `forward` must use exactly the layers you created in `__init__`.

# 6. Check your model shape and size

The model must not exceed 5M trainable parameters.


In [ ]:
# device refers to the GPU or the CPU, depending whether GPU is available.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Call our model. The number of output classes is passed as an argument.
# Re-import the ConvNN
# We need to always re-import the ConvNN after importlib reload as autoreload do not work in Google Colab as per https://github.com/googlecolab/colabtools/issues/5580.
# If you work without Google Colab, you will not need this.
importlib.reload(models)
from models import ConvNN

model = ConvNN(num_classes=trainset.num_classes).to(device)

# display information about the model
print(summary(model, input_size=(64, 3, 32, 32)))

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable parameters:', params)
if params > 5_000_000:
    raise Exception('Your model is unecessarily complex, scale down!')

🔍 **Reading the `torchinfo` summary.** `input_size=(64, 3, 32, 32)` means a dummy batch of 64 images. Each row shows one layer's output shape (including the batch dimension) and its parameter count. Check that:
- the last row is `[64, 10]`
- trainable parameters equal **2,198,602**
- the shapes halve after each `MaxPool2d`

If the shapes do not line up (for example a `mat1 and mat2 shapes cannot be multiplied` error), the input size of `fc1` is wrong. The helper below prints each layer's output shape in the order the layers run.

In [ ]:
# Optional helper added for learning: trace tensor shapes through the network
hooks = [m.register_forward_hook(lambda mod, inp, out, n=n: print(f"{n:8s} -> {tuple(out.shape)}"))
         for n, m in model.named_children()]
model.eval()
with torch.no_grad():
    model(torch.zeros(2, 3, 32, 32, device=device))
for h in hooks:
    h.remove()

# 7.  Monitoring


Don't edit this cell 😺

In [ ]:
# Custom callback for plotting loss and accuracy during training

class PlotLogAccuracy:

  def __init__(self):
    self.epochs = []
    self.train_losses = []
    self.val_losses = []
    self.train_acc = []
    self.val_acc = []
    self.epoch_count = 0

  def update(self, train_loss, train_acc, val_loss, val_acc):
    self.epochs.append(self.epoch_count)
    self.train_losses.append(train_loss)
    self.val_losses.append(val_loss)
    self.train_acc.append(train_acc)
    self.val_acc.append(val_acc)
    self.epoch_count += 1

    clear_output(wait=True)
    plt.figure(figsize=(16, 6))
    plt.subplot(121)
    plt.plot(self.epochs, self.train_losses, label="train loss")
    plt.plot(self.epochs, self.val_losses, label="validation loss")
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.title('Model Loss')
    plt.legend()
    plt.subplot(122)
    plt.plot(self.epochs, self.train_acc, label="training accuracy")
    plt.plot(self.epochs, self.val_acc, label="validation accuracy")
    plt.legend()
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.title('Model Accuracy')
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.show();

💡 **Monitoring (same as Lab 4).** After each epoch the plotter redraws the training and validation loss and accuracy curves.
- With no augmentation, expect the **training accuracy to overtake validation** after a few epochs.
- The important signal is **validation loss**. If it starts rising while training loss keeps falling, the model is over-fitting and further epochs will not help (handout-05 slides 58–60).

# 8. Optimiser and Dataloader Hyperparameter Setup

Here we define the dataloaders which handle the dataset, and the training optimiser.

Their hyperparameters include (but are not limited to):


*   Number of training epochs
*   Batch size
* Learning rate (step size)
* Optimiser momentum
* Dataset shuffling

Remember from class that the optimiser sets the weights and biases of the network during training. [Many exist](https://docs.pytorch.org/docs/stable/optim.html).


In [ ]:
epochs = 30           # shorter for notebook demo - increase for training
batch_size = 64       # play with this
learning_rate = 0.01  # play with this
momentum = 0.9        # play with this

criterion = nn.CrossEntropyLoss()

# Reload ConvNN
importlib.reload(models)
from models import ConvNN

# setting the model (calling this again will reset the weights)
model = ConvNN(num_classes=trainset.num_classes).to(device)

# play with this
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum, weight_decay=5e-4)

trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True,
                         num_workers=0, pin_memory=torch.cuda.is_available())
valloader   = DataLoader(valset, batch_size=batch_size, shuffle=False,
                         num_workers=0, pin_memory=torch.cuda.is_available())

plotter = PlotLogAccuracy()

train_loss_hist, val_loss_hist = [], []
train_acc_hist, val_acc_hist = [], []

print('Setup complete.')

### Explanation: Optimiser and hyperparameters

**Objective.** Choose the optimiser and training hyperparameters so that `ConvNN` reaches the lab target of 80% validation accuracy.

**What was modified.** Two changes were made to the provided setup: `epochs` was increased from 10 to 30, and `weight_decay=5e-4` was added to `torch.optim.SGD`. The batch size (64), learning rate (0.01) and momentum (0.9) were left unchanged.

**Method.** The training objective is the cross-entropy loss on the logits $z$:
$$\mathcal{L} = -\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{k} e^{z_{n,k}}}$$
SGD with momentum and weight decay, as implemented in PyTorch, updates each parameter $\theta$ using the mini-batch gradient $g_t$:
$$g_t \leftarrow g_t + \lambda\theta_t, \qquad v_{t+1} = \mu v_t + g_t, \qquad \theta_{t+1} = \theta_t - \eta\,v_{t+1}$$
Here $\eta = 0.01$, $\mu = 0.9$ and $\lambda = 5\times10^{-4}$. Adding $\lambda\theta$ to the gradient is equivalent to minimising $\mathcal{L} + \frac{\lambda}{2}\lVert\theta\rVert^2$, i.e. L2 regularisation.

**Why it matters.** Momentum accumulates past gradients, which smooths out the noise from 64-sample mini-batches and speeds up progress in consistent directions. Weight decay penalises large weights and favours simpler functions, which reduces overfitting. Without augmentation, training accuracy otherwise rises well above validation accuracy. Ten epochs is not enough for this network to converge, so training was extended to 30 epochs. The training loader is shuffled every epoch so the mini-batches are decorrelated. The validation loader is not shuffled because order does not affect evaluation.

**How the notebook uses it.** Running this cell re-creates the model with fresh weights, along with the optimiser and data loaders. Section 9 trains for `epochs` epochs. After each epoch it computes validation loss and accuracy in `model.eval()` mode under `torch.no_grad()`, and `PlotLogAccuracy` plots the training and validation curves, so any overfitting shows as a growing gap between them. Section 11 reloads the saved `model.pth` and reports its validation accuracy against the 80% objective. When this configuration was tested, validation accuracy passed 80% after about 8 epochs and finished at roughly 85–87%.

### 💡 SGD + momentum vs Adam (complementing the explanation above)

| | Lab 4 | Lab 5 |
|---|---|---|
| optimiser | Adam, lr 1e-3 | **SGD + momentum 0.9, lr 0.01** |
| weight decay | 1e-4 | 5e-4 |
| LR schedule | cosine annealing | **none (constant)** |
| batch size / epochs | 128 / 40 | 64 / 30 → ⌈45000/64⌉ = **704 updates per epoch** |
| augmentation | random horizontal flip | none |

- **Why SGD with momentum for CNNs?** Adam usually converges faster at the start. Well-tuned SGD with momentum often *generalises* slightly better on image classification, which is why most CNN benchmarks use it (handout-07 slide 31). BatchNorm makes the relatively high learning rate of 0.01 safe.
- **Momentum intuition** (handout-05 slide 51). The velocity $v$ is a running sum of past gradients. Consistent directions build up speed, and oscillating directions cancel out, like a heavy ball rolling down the loss surface.
- **The missing schedule.** With a constant learning rate, the validation accuracy typically flattens and jitters in the final epochs. Adding `CosineAnnealingLR(optimizer, T_max=epochs)` (with `scheduler.step()` once per epoch), or a one-cycle policy, usually adds a few points (handout-07 slides 32–33).

# 9. Training and Validation

The following cell contains the main training and validation loop for the neural network.



In [ ]:
print('Starting training...')

# Epoch Loop: It iterates through the specified number of training epochs.

for epoch in range(epochs):

    # For each epoch, all data in the dataset is processed.

    model.train() # Sets the model to training mode

    running_loss, correct, total = 0.0, 0, 0 # resetting loss/accuracy metrics

    #  Iterates through the `trainloader` to get mini-batches of data
    for x_batch, y_batch in tqdm(trainloader, desc=f'Epoch {epoch+1}/{epochs} [Train]', leave=False):

        # copying data to the GPU (the `device`) if GPU is available
        x_batch, y_batch = x_batch.to(device), y_batch.to(device).view(-1)

        optimizer.zero_grad()           # setting up gradient to zero
        out = model(x_batch)            # Performs the forward pass
        loss = criterion(out, y_batch)  # calculates the loss
        loss.backward()                 # performs backpropagation in parallel on the batch
        optimizer.step()                # optimizer steps to update the model's weights

        # updating current training Loss and Accuracy on the mini-batch
        running_loss += loss.item() * x_batch.size(0)
        preds = out.argmax(1)
        total += y_batch.size(0)
        correct += (preds == y_batch).sum().item()

    train_loss = running_loss / len(trainloader.dataset)
    train_acc = correct / total

    # Computing Validation Loss and Accuracy

    model.eval() # switching model to eval mode, disabling dropout/batchnorm/other custom modules
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad(): # we disable gradient computation to save some memory
        for x_batch, y_batch in tqdm(valloader, desc=f'Epoch {epoch+1}/{epochs} [Val]', leave=False):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device).view(-1)
            out = model(x_batch)
            loss = criterion(out, y_batch)
            val_running_loss += loss.item() * x_batch.size(0)
            preds = out.argmax(1)
            val_total += y_batch.size(0)
            val_correct += (preds == y_batch).sum().item()
    val_loss = val_running_loss / len(valloader.dataset)
    val_acc = val_correct / val_total

    # Record training loss and accuracy
    train_loss_hist.append(train_loss); val_loss_hist.append(val_loss)
    train_acc_hist.append(train_acc); val_acc_hist.append(val_acc)

    # At the end of the epoch, plot loss and accuracy for training/validation
    plotter.update(train_loss, train_acc, val_loss, val_acc)
    print(f'Epoch {epoch+1}: Train Loss {train_loss:.4f} Acc {train_acc:.4f} | Val Loss {val_loss:.4f} Acc {val_acc:.4f}')

print('Training finished.')


### 🛠️ The training loop: what is the same and what differs from Lab 4

- **Same five core steps:** `zero_grad` → forward → loss → `backward` → `step`. The validation pass also runs under `model.eval()` + `torch.no_grad()`, exactly as in Lab 4 (see Lab 4's line-by-line table).
- **No flip augmentation** and **no `scheduler.step()`**.
- **`model.train()` vs `model.eval()` matters even more here.** `BatchNorm2d` normalises each channel over batch × height × width. In eval mode it switches to the running averages accumulated during training, so predictions no longer depend on which other images happen to be in the batch.
- **Backpropagation through convolutions** works by the same chain rule. A shared filter's gradient is the **sum** of its gradients at every position where it was applied (handout-06 slides 7–9).

**Expected result.** This notebook contains **no saved training output**. The explanation cell above reports that, when this configuration was tested, validation accuracy passed 80% after about 8 epochs and finished at roughly 85–87%. When you run it:
- Expect training accuracy to climb well above validation accuracy in the later epochs. That gap is over-fitting, and it is limited by dropout and weight decay.
- If validation accuracy peaks and then *falls*, consider fewer epochs or the improvements below.

# 10. Save your model to your lab 5 directory.

In [ ]:
MODEL_PATH = 'model.pth'  # DO NOT CHANGE THE NAME, this is required by the evaluation script

# Save the entire model
torch.save(model, os.path.join('/content/gdrive/MyDrive/4c16-labs/code/dist/lab-05', MODEL_PATH))
print('Model saved to', MODEL_PATH)


💡 **Saving and loading.** This works as in Lab 4: the whole model is pickled to `model.pth`, which is the name the grader requires. The save cell writes to the Drive `lab-05` folder, and the load cell reads `MODEL_PATH` relative to the current directory. These are the same folder only because of the `%cd` at the top, so remember this if you run locally. Reloading with `loaded.eval()` should reproduce the final epoch's validation accuracy exactly.

# 11. Load and test your model

Note: we will evaluate your model on a different dataset. This cell is to demonstrate model loading and give you a rough idea of your model performance.


In [ ]:
loaded = torch.load(MODEL_PATH, map_location=device, weights_only=False)
loaded.eval()
val_loss_sum, val_correct, val_total = 0.0, 0, 0
with torch.no_grad():
    for x_batch, y_batch in valloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device).view(-1)
        out = loaded(x_batch)
        loss = criterion(out, y_batch)
        val_loss_sum += loss.item() * x_batch.size(0)
        preds = out.argmax(1)
        val_total += y_batch.size(0)
        val_correct += (preds == y_batch).sum().item()
final_val_loss = val_loss_sum / len(valloader.dataset)
final_val_acc = val_correct / val_total
print(f'Reloaded Model - Val Loss: {final_val_loss:.4f}, Val Acc: {final_val_acc:.4f}')

🔍 **Which classes are hard?** The helper below breaks validation accuracy down per class, for whichever `model` is currently in memory. On CIFAR-10, **cat, dog and bird** are typically the hardest (similar shapes, fur and textures). Vehicles (automobile, ship, truck) are usually the easiest.

In [ ]:
# Optional helper added for learning: per-class validation accuracy
model.eval()
n = trainset.num_classes
correct, count = torch.zeros(n), torch.zeros(n)
with torch.no_grad():
    for xb, yb in valloader:
        xb, yb = xb.to(device), yb.to(device).view(-1)
        preds = model(xb).argmax(1)
        for c in range(n):
            mask = yb == c
            count[c] += mask.sum().item()
            correct[c] += (preds[mask] == c).sum().item()
for c in range(n):
    print(f"{str(trainset.labels[c]):12s} {correct[c] / count[c]:.3f}")

# Lab objective: get 80% accuracy

### 🛠️ How to reach (and go beyond) 80%

The current setup should already pass. If yours does not, or you want more accuracy, change **one** thing at a time, from the most effective to the least:

| Idea | How | Handout |
|---|---|---|
| **Data augmentation** | per image: random crop (pad 4, crop back to 32) plus horizontal flip. For example, in the loop: `flip = torch.rand(x_batch.size(0), 1, 1, 1, device=device) < 0.5` then `x_batch = torch.where(flip, x_batch.flip(3), x_batch)` | 07 slides 26–27 |
| **LR schedule** | `CosineAnnealingLR` or `OneCycleLR` | 07 slides 32–33 |
| **More epochs** | only once augmentation stops the over-fitting | 05 slide 48 |
| **Global average pooling** | replace flatten + `fc1` with `AdaptiveAvgPool2d(1)` + `Linear(256, 10)`. This saves about 1M parameters and reduces over-fitting | 06 slides 30–31 |
| **Residual (skip) connections** | `out = F.relu(x + block(x))` lets much deeper networks train (ResNet) | 07 slides 16–24 |
| **AdamW** | `AdamW(lr=1e-3, weight_decay=5e-2)` is a strong alternative to SGD | 07 slide 31 |
| **He initialisation** | PyTorch's default for `Conv2d`/`Linear` is already Kaiming-uniform, so nothing to do | 07 slides 28–29 |

**Rule of thumb.** If training accuracy is ≫ validation accuracy, add regularisation or augmentation. If both are low, make the model bigger or train longer.

---

### ✅ Key takeaways
- Convolutions use **local connectivity** and **weight sharing**, giving huge parameter savings and translation equivariance compared with dense layers.
- Parameter counts: Conv2d = $C_\text{in} C_\text{out} k^2 + C_\text{out}$; BatchNorm = $2C$; pooling, ReLU and dropout have none.
- Output size $= \lfloor (n + 2p - k)/s \rfloor + 1$. A 3×3 convolution with padding 1 keeps the size, and 2×2 pooling halves it.
- The standard design stacks conv–BN–ReLU blocks, pools, doubles the channels, and finishes with a small classifier head that outputs logits.
- Depth grows the receptive field. Deep features represent increasingly abstract parts of the image.
- BatchNorm, dropout, weight decay, augmentation and LR schedules are the main levers for going from "trains" to "generalises".

### ❓ Self-check
1. How many parameters does `nn.Conv2d(64, 128, 3)` have? And `nn.Conv2d(64, 128, 5)`?
2. What is the output shape of `Conv2d(3, 16, 5, stride=1, padding=0)` on a 32×32 image? With `stride=2, padding=2`?
3. If you add a fourth MaxPool before flattening, what must `fc1`'s input size become?
4. Why does a CNN with 2.2M parameters beat a dense network with 3.8M parameters on CIFAR-10?
5. Validation accuracy peaks at epoch 12 and then slowly drops while training accuracy reaches 99%. What is happening, and what would you change?

<details><summary>Answers</summary>

1. 64·128·9 + 128 = 73,856; 64·128·25 + 128 = 204,928.
2. ⌊(32 − 5)/1⌋ + 1 = 28, giving (16, 28, 28). ⌊(32 + 4 − 5)/2⌋ + 1 = 16, giving (16, 16, 16).
3. 256·2·2 = 1024.
4. Weight sharing and locality match the structure of images (the same features appear anywhere in the image), so every parameter is reused across all positions. The MLP must learn each feature separately at every position and over-fits.
5. Over-fitting. Add augmentation or more dropout/weight decay, add an LR schedule, or stop early at the best validation epoch.
</details>